In [42]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load data
df = pd.read_csv('../data/processed/telco_features_CatBoost.csv')  # Your new dataset

# 2. Define features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()  # Automatically detect categorical columns

# categorical_cols = [
#     'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
#     'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
#     'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
#     'PaperlessBilling', 'PaymentMethod', 'tenure_group'
# ]

target_col = 'Churn'
feature_cols = [col for col in df.columns if col not in [target_col]]

# 3. Prepare X and y
X = df[feature_cols]
y = df[target_col]

# 4. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Get categorical column indices (CatBoost needs indices, not names)
cat_features_indices = [i for i, col in enumerate(feature_cols) if col in categorical_cols]

# 6. Create CatBoost Pool (optional but recommended)
train_pool = Pool(X_train, y_train, cat_features=cat_features_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_features_indices)

# 7. Train CatBoost
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function='Logloss',
    cat_features=cat_features_indices,  # KEY PARAMETER
    verbose=50,
    random_state=42,
    #scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1])  # Handle imbalance (reùoved cuz the churn is already balanced)
)

model.fit(
    train_pool,
    eval_set=test_pool,
    early_stopping_rounds=50,
    use_best_model=True
)

# 8. Evaluate
y_pred = model.predict(test_pool)
y_proba = model.predict_proba(test_pool)[:, 1]

print("="*60)
print("CATBOOST PERFORMANCE")
print("="*60)
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

# 9. Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

0:	learn: 0.6518251	test: 0.6516809	best: 0.6516809 (0)	total: 206ms	remaining: 1m 43s
50:	learn: 0.5060211	test: 0.5074976	best: 0.5074420 (42)	total: 12.4s	remaining: 1m 49s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5074419581
bestIteration = 42

Shrink model to first 43 iterations.
CATBOOST PERFORMANCE
AUC-ROC: 0.787

Classification Report:
              precision    recall  f1-score   support

    Retained       0.88      0.66      0.75      6542
     Churned       0.70      0.90      0.78      5745

    accuracy                           0.77     12287
   macro avg       0.79      0.78      0.77     12287
weighted avg       0.79      0.77      0.77     12287


Top 10 Most Important Features:
              feature  importance
14           Contract   94.160826
13    StreamingMovies    0.626501
4              tenure    0.591503
19   value_risk_ratio    0.558750
17     MonthlyCharges    0.558216
20  avg_monthly_spend    0.547682
15   PaperlessBilling    0.532